In [4]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn, FasterRCNN_MobileNet_V3_Large_320_FPN_Weights
import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights


# ================= CONFIG =================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASSES = [
    'nevus', 'melanoma', 'seborrheic keratosis', 'pigmented benign keratosis', 
    'vascular lesion', 'basal cell carcinoma', 'squamous cell carcinoma', 
    'dermatofibroma', 'actinic keratosis'
]

num_classes = len(CLASSES) + 1

MODEL_PATH = r"D:\xu_li_anh\btl\checkpoin\models\best_model.pth"
IMG_PATH   = r"D:\xu_li_anh\btl\data\test\ISIC_0033458_jpg.rf.6566dbc0325a79ca60cb40a1322097d5.jpg"   # đổi ảnh test của bạn

# ================= TRANSFORM =================
transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# ================= LOAD MODEL =================

# 5. Khởi tạo Mô hình (FIX MODEL)
model = fasterrcnn_mobilenet_v3_large_320_fpn(weights=FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT)
in_channels = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_channels, num_classes)
model.to(DEVICE)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

# ================= LOAD IMAGE =================
image = cv2.imread(IMG_PATH)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

orig_h, orig_w = image.shape[:2]

aug = transform(image=image)
image_tensor = aug["image"].to(DEVICE)

# Faster R-CNN cần list input
with torch.no_grad():
    outputs = model([image_tensor])

# ================= POST PROCESS =================
boxes = outputs[0]["boxes"].cpu().numpy()
scores = outputs[0]["scores"].cpu().numpy()
labels = outputs[0]["labels"].cpu().numpy()

# threshold
CONF_THRESHOLD = 0.7

for box, score, label in zip(boxes, scores, labels):
    if score < CONF_THRESHOLD:
        continue

    x1, y1, x2, y2 = box.astype(int)

    class_name = CLASSES[label - 1]  # label bắt đầu từ 1

    cv2.rectangle(image, (x1, y1), (x2, y2), (0,255,0), 2)
    cv2.putText(
        image,
        f"{class_name} {score:.2f}",
        (x1, y1 - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0,255,0),
        2
    )

# ================= SHOW RESULT =================
cv2.imshow("Detection", cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
cv2.waitKey(0)
cv2.destroyAllWindows()